# Advanced Features

These code snippets demonstrate more advanced configuration and control of a PULSAR HRI actuator. It builds upon the basic example so you can upgrade it by adding these features.

In this notebook you will learn how to:

- Change the actuator PCP address
- Tune performance profiles for torque and speed
- Set custom control parameters such as stiffness and damping
- Optionally save configuration to persistent memory

## Import necessary modules


In [ ]:
from pcp_api import PCP_over_USB, PulsarActuator


## Instantiate and connect the actuator

These advanced features operate on a connected `actuator` instance. If you are following the basic example, you can reuse that setup. The cell below is included so this notebook can also be run on its own.


In [ ]:
port = PCP_over_USB.get_port()  # Auto-detect the serial port
# port = "COM1"
print(f"Connecting to {port}")
adapter = PCP_over_USB(port)

ACTUATOR_ADDRESS = 0  # 0 for direct USB connection, or use the actuator address if using CAN
actuator = PulsarActuator(adapter, ACTUATOR_ADDRESS)

if not actuator.connect():
    print(f"Could not connect to the actuator {actuator.address}")
    adapter.close()
    raise SystemExit(1)

print(f"Connected to the actuator {actuator.address} (model: {actuator.model}, firmware: {actuator.firmware_version})")


## Change the actuator address

Set a new address. This is stored in the actuator persistent memory, so it is retained across power cycles. For actuators, we recommend using addresses starting at `0x10` because lower addresses are reserved for adapters and special purposes.


In [ ]:
actuator.changeAddress(0x15)


## Change performance profiles for torque and speed

The available performance profiles are defined in [`PulsarActuator.TorquePerformance`](class_PulsarActuator.md#pcp_api.pulsar_actuator.PulsarActuator.TorquePerformance) and [`PulsarActuator.SpeedPerformance`](class_PulsarActuator.md#pcp_api.pulsar_actuator.PulsarActuator.SpeedPerformance).


In [ ]:
actuator.set_torque_performance(PulsarActuator.TorquePerformance.AGGRESSIVE)
actuator.set_speed_performance(PulsarActuator.SpeedPerformance.BALANCED)


## Set custom control parameters

The available parameters are defined in [`PulsarActuator.PCP_Parameters`](class_PulsarActuator.md#pcp_api.pulsar_actuator.PulsarActuator.PCP_Parameters). The `set_parameters` method takes a dictionary where the keys are parameter names and the values are the desired settings.


In [ ]:
actuator.set_parameters({
    PulsarActuator.PCP_Parameters.K_DAMPING: 7.7,
    PulsarActuator.PCP_Parameters.K_STIFFNESS: 8.8,
})


## Read back parameters

This reads back the current configuration from the actuator. In this case, the `get_parameters` method takes a list of parameters instead of a dictionary. You can also use `get_parameters_all()` to read all parameters at once.


In [ ]:
params = actuator.get_parameters([
    PulsarActuator.PCP_Parameters.MODE,
    PulsarActuator.PCP_Parameters.SETPOINT,
    PulsarActuator.PCP_Parameters.K_STIFFNESS,
])
# params = actuator.get_parameters_all()
print(params)
print(params[PulsarActuator.PCP_Parameters.K_STIFFNESS])


## Save the configuration to persistent memory

This saves the current parameters to the actuator persistent memory so they are retained across power cycles.


In [ ]:
actuator.save_config()


## Cleanup

Disconnect the actuator and close the adapter when you are done.


In [ ]:
actuator.disconnect()
adapter.close()
